# Section 5 Onboarding Notebook: Correlated Wavefunction Methods — Configuration Interaction (CI), MP2, and Coupled Cluster (CCSD)

### Overview & Pedagogical Design
In **Section 4 (`04_hartree_fock_scf_notebook.ipynb`)**, you built a Restricted Hartree–Fock (RHF) solver from scratch to obtain the single-determinant reference state $|\Phi_0\rangle$, canonical molecular orbital (MO) coefficients $\mathbf{C}$, orbital eigenvalues $\boldsymbol{\varepsilon}$, and total RHF energy $E_{\text{RHF}}$. Because RHF treats electron–electron repulsion only as an averaged mean field, it misses the **electron correlation energy** ([Szabo & Ostlund](https://pycrawfordprogproj.readthedocs.io/en/latest/Project_04/Project_04.html)):

$$E_{\text{corr}} = E_{\text{exact}} - E_{\text{RHF}} < 0$$

In this notebook, **no correlated method implementations are pre-written for you**. Instead, you will implement every single building block of **AO-to-MO integral transformations**, **Second-Order Møller–Plesset Perturbation Theory (MP2 & SCS-MP2)**, **Configuration Interaction (CISD & Full CI)**, and **Coupled Cluster Singles and Doubles (CCSD)** across **20 granular `# TODO` functions**:

1. **Part 1 — AO-to-MO Integral Transformations & Spin-Orbital Basis (TODOs 1–4)**:
   - `transform_1e_ao_to_mo`: Transform $\mathbf{H}^{\text{core}}$ to $\mathbf{h}^{\text{MO}}$.
   - `transform_eri_ao_to_mo_quarter_steps`: Implement the four sequential $\mathcal{O}(N^5)$ quarter-transformations explicitly (no black-box 4-index one-liner!) and compare its flop count to a naive $\mathcal{O}(N^8)$ transform.
   - `build_spin_orbital_integrals`: Map spatial orbitals $p \mapsto (2p\,\alpha, 2p+1\,\beta)$, transpose chemist's $(pr|qs)$ to physicist's $\langle pq | rs \rangle$, and construct the antisymmetrized tensor $\langle pq \| rs \rangle = \langle pq | rs \rangle - \langle pq | sr \rangle$.
   - `compute_spin_orbital_rhf_energy`: Reconstruct $E_{\text{RHF}}$ purely from spin-orbital integrals $h_{pq}$ and $\langle pq \| rs \rangle$ to verify your spin-orbital basis.
2. **Part 2 — Møller–Plesset Perturbation Theory: MP2, SCS-MP2 & Correlated 1-RDM (TODOs 5–8)**:
   - `build_mp2_denominators`: Construct the 4D energy denominator tensor $\Delta_{iajb} = \varepsilon_i + \varepsilon_j - \varepsilon_a - \varepsilon_b$ via NumPy broadcasting.
   - `compute_spatial_mp2_and_scs`: Compute first-order amplitudes $t_{ij}^{ab}$, separate opposite-spin ($E_{\text{OS}}$) and same-spin ($E_{\text{SS}}$) correlation energies ([Psi4NumPy MP2](https://github.com/psi4/psi4numpy/blob/master/Tutorials/05_Moller-Plesset/5a_conventional-mp2.ipynb)), evaluate Grimme's SCS-MP2 energy, and build the occupied pair-correlation matrix $e_{ij}$.
   - `compute_spin_orbital_mp2`: Independently evaluate $E_{\text{corr}}^{\text{MP2}} = \frac{1}{4}\sum_{ijab} |\langle ij \| ab \rangle|^2 / \Delta_{ij}^{ab}$ in the spin-orbital basis.
   - `compute_mp2_1rdm_and_noons`: Contract $t_{ij}^{ab}$ to form the unrelaxed MP2 one-particle density matrix $\mathbf{P}^{\text{MP2}}$ and diagonalize it for **Natural Orbital Occupation Numbers (NOONs)**.
3. **Part 3 — Configuration Interaction: Determinants, Slater–Condon Rules & CISD (TODOs 9–13)**:
   - `generate_sz0_cisd_determinants`: Enumerate $|\Phi_0\rangle$, spin-conserving single excitations $|\Phi_i^a\rangle$, and spin-conserving double excitations $|\Phi_{ij}^{ab}\rangle$ as sorted spin-orbital tuples.
   - `align_determinants_and_parity`: Compute the differing spin-orbitals, common spin-orbitals, excitation rank, and exact fermionic permutation sign $(-1)^P$ between any two determinants $|D_I\rangle$ and $|D_J\rangle$.
   - `slater_condon_element`: Implement all four cases ($0, 1, 2, \ge 3$ orbital differences) of the Slater–Condon rules.
   - `run_cisd_solver`: Assemble $\mathbf{H}_{\text{CISD}}$, verify **Brillouin's theorem** ($\langle \Phi_0 | \hat{H} | \Phi_i^a \rangle = 0$), and diagonalize for $E_{\text{CISD}}$ and $|c_0|^2$.
   - `solve_2e_singlet_fci`: Construct and diagonalize the exact 2-electron singlet Full CI Hamiltonian and extract the correlated 1-RDM and 2-RDM.
4. **Part 4 — Coupled Cluster Singles & Doubles: Stanton–Gauss Spin-Orbital CCSD (TODOs 14–19)**:
   - `build_ccsd_tau_tensors`: Construct $\tilde{\tau}_{ij}^{ab}$ and $\tau_{ij}^{ab}$ ([Crawford Project 05](https://pycrawfordprogproj.readthedocs.io/en/latest/Project_05/Project_05.html)).
   - `build_ccsd_F_intermediates`: Construct the three 1-body intermediates $\mathcal{F}_{ae}, \mathcal{F}_{mi}, \mathcal{F}_{me}$.
   - `build_ccsd_W_intermediates`: Construct the three 2-body intermediates $\mathcal{W}_{mnij}, \mathcal{W}_{abef}, \mathcal{W}_{mbej}$.
   - `update_t1_amplitudes`: Evaluate the 7 terms of the CCSD Singles residual and update $t_i^a$.
   - `update_t2_amplitudes`: Evaluate the CCSD Doubles residual (including explicit permutation operators $\hat{P}(ab)$ and $\hat{P}(ij)$) and update $t_{ij}^{ab}$.
   - `run_ccsd_driver`: Assemble the full iterative CCSD solver, evaluate $E_{\text{corr}}^{\text{CCSD}}$, and compute the Lee–Taylor $T_1$ multireference diagnostic ([Psi4NumPy CCSD](https://github.com/psi4/psi4numpy/blob/master/Coupled-Cluster/Spin_Orbitals/CCSD/CCSD_DIIS.py)).
5. **Part 5 — Size-Extensivity Scaling & Diagnostic Plots (TODO 20)**:
   - `compute_size_extensivity_scaling` & `plot_correlated_diagnostics`: Prove mathematically and numerically why MP2 and CCSD scale linearly with monomer count $M$, whereas truncated CISD degrades as $M^{-1/2}$.
6. **Part 6 — Step-by-Step Checkpoint & Autograding Suite**:
   - Run `run_all_checkpoints()` as you complete each part to test each function individually against exact analytical benchmarks on $\text{HeH}^+$ and $\text{H}_2\text{O}$ (`STO-3G`).

In [ ]:
import sys
sys.path.insert(0, ".")
import numpy as np
from scipy.linalg import eigh
import matplotlib.pyplot as plt
from rhf_solver import run_rhf

np.set_printoptions(precision=6, suppress=True, linewidth=110)
plt.rcParams.update({
    "figure.figsize": (12, 4.2),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

# Load precomputed STO-3G AO integrals from Section 4 for HeH+ (2e, 2 AOs) and H2O (10e, 7 AOs)
heh_data = np.load("heh_plus_sto3g_integrals.npz")
h2o_data = np.load("h2o_sto3g_integrals.npz")

rhf_heh = run_rhf(
    heh_data["S"], heh_data["T"], heh_data["V"], heh_data["eri"],
    float(heh_data["e_nuc"]), int(heh_data["n_elec"]), e_tol=1e-12, d_tol=1e-10
)
rhf_h2o = run_rhf(
    h2o_data["S"], h2o_data["T"], h2o_data["V"], h2o_data["eri"],
    float(h2o_data["e_nuc"]), int(h2o_data["n_elec"]), e_tol=1e-12, d_tol=1e-10
)

# Canonicalize final MOs against the converged Fock matrix
for data_dict, rhf_obj in ((heh_data, rhf_heh), (h2o_data, rhf_h2o)):
    H_c = data_dict["T"] + data_dict["V"]
    J_c = np.einsum("uvls,ls->uv", data_dict["eri"], rhf_obj.density_matrix, optimize=True)
    K_c = np.einsum("ulvs,ls->uv", data_dict["eri"], rhf_obj.density_matrix, optimize=True)
    F_c = H_c + 2.0 * J_c - K_c
    evals_s, evecs_s = eigh(data_dict["S"])
    X_c = (evecs_s / np.sqrt(evals_s)) @ evecs_s.T
    eps_c, C_p = eigh(X_c.T @ F_c @ X_c)
    rhf_obj.orbital_energies = eps_c
    rhf_obj.mo_coeff = X_c @ C_p

print(f"Loaded Section 4 RHF References:")
print(f"  HeH+ (STO-3G, 2e, 2 AOs) : E_RHF = {rhf_heh.e_total:.10f} Eh")
print(f"  H2O  (STO-3G, 10e, 7 AOs): E_RHF = {rhf_h2o.e_total:.10f} Eh")

## Part 1: AO-to-MO Integral Transformation & Antisymmetrized Spin-Orbital Basis

### Mathematical Specifications for TODOs 1–4
1. **One-Electron MO Transformation (`transform_1e_ao_to_mo`)**:
   Given the $(N_{\text{AO}}, N_{\text{AO}})$ core Hamiltonian $\mathbf{H}^{\text{core}} = \mathbf{T} + \mathbf{V}$ and canonical MO coefficient matrix $\mathbf{C}$ (where column $p$ is MO $\phi_p$), compute:
   $$h_{pq} = \sum_{\mu\nu} C_{\mu p}\,H_{\mu\nu}^{\text{core}}\,C_{\nu q}$$

2. **Sequential $\mathcal{O}(N^5)$ Quarter-Transformations for 2-Electron ERIs (`transform_eri_ao_to_mo_quarter_steps`)**:
   A naive 8-index sum $(pq|rs) = \sum_{\mu\nu\lambda\sigma} C_{\mu p} C_{\nu q} C_{\lambda r} C_{\sigma s} (\mu\nu|\lambda\sigma)$ requires $N_{\text{AO}}^8$ operations. Instead, you must perform **four sequential single-index contractions** ([Crawford Project 04](https://pycrawfordprogproj.readthedocs.io/en/latest/Project_04/Project_04.html)), each costing $N_{\text{AO}}^5$ operations:
   $$\begin{aligned}
   I^{(1)}_{p\nu\lambda\sigma} &= \sum_\mu C_{\mu p}\,(\mu\nu|\lambda\sigma) \\
   I^{(2)}_{pq\lambda\sigma} &= \sum_\nu C_{\nu q}\,I^{(1)}_{p\nu\lambda\sigma} \\
   I^{(3)}_{pqr\sigma} &= \sum_\lambda C_{\lambda r}\,I^{(2)}_{pq\lambda\sigma} \\
   (pq|rs) &= \sum_\sigma C_{\sigma s}\,I^{(3)}_{pqr\sigma}
   \end{aligned}$$

3. **Spin-Orbital Basis Construction (`build_spin_orbital_integrals`)**:
   Map each spatial orbital $p \in \{0, \dots, N_{\text{MO}}-1\}$ to two spin orbitals:
   - Even index $2p$: $\alpha$-spin orbital $\psi_{2p}(\mathbf{x}) = \phi_p(\mathbf{r})\alpha(\omega)$ with energy $\varepsilon_{2p}^{\text{SO}} = \varepsilon_p$
   - Odd index $2p+1$: $\beta$-spin orbital $\psi_{2p+1}(\mathbf{x}) = \phi_p(\mathbf{r})\beta(\omega)$ with energy $\varepsilon_{2p+1}^{\text{SO}} = \varepsilon_p$

   Convert chemist's spatial integrals $(pr|qs)$ (where $p,r$ are electron 1 and $q,s$ are electron 2) into **physicist's Dirac spin-orbital integrals** $\langle pq | rs \rangle = \int \psi_p^*(\mathbf{x}_1)\psi_q^*(\mathbf{x}_2) r_{12}^{-1} \psi_r(\mathbf{x}_1)\psi_s(\mathbf{x}_2)\,d\mathbf{x}_1 d\mathbf{x}_2$:
   $$\langle pq | rs \rangle = (pr|qs)\,\delta_{\sigma_p \sigma_r}\,\delta_{\sigma_q \sigma_s}$$
   Then form the **antisymmetrized spin-orbital integral tensor**:
   $$\langle pq \| rs \rangle = \langle pq | rs \rangle - \langle pq | sr \rangle$$

4. **Spin-Orbital RHF Energy Verification (`compute_spin_orbital_rhf_energy`)**:
   Verify that the occupied spin orbitals $i, j \in \{0, \dots, N_{\text{elec}}-1\}$ reproduce the exact RHF total energy:
   $$E_{\text{RHF}} = \sum_{i=0}^{N_{\text{elec}}-1} h_{ii}^{\text{SO}} + \frac{1}{2}\sum_{i=0}^{N_{\text{elec}}-1}\sum_{j=0}^{N_{\text{elec}}-1} \langle ij \| ij \rangle + E_{\text{nuc}}$$

In [ ]:
def transform_1e_ao_to_mo(H_core: np.ndarray, C: np.ndarray) -> np.ndarray:
    '''
    TODO 1: Transform the 1-electron core Hamiltonian from the AO basis to the spatial MO basis.

    Parameters
    ----------
    H_core : np.ndarray, shape (n_ao, n_ao)
        One-electron core Hamiltonian matrix T + V in the AO basis.
    C : np.ndarray, shape (n_ao, n_mo)
        Canonical RHF molecular orbital coefficient matrix (columns are MOs).

    Returns
    -------
    h_mo : np.ndarray, shape (n_mo, n_mo)
        One-electron core Hamiltonian in the spatial MO basis.
    '''
    # TODO 1: Implement h_mo = C^T H_core C
    raise NotImplementedError("Complete TODO 1: transform_1e_ao_to_mo")


def transform_eri_ao_to_mo_quarter_steps(eri_ao: np.ndarray, C: np.ndarray) -> np.ndarray:
    '''
    TODO 2: Transform the 4-index AO ERI tensor (uv|ls) into the spatial MO basis (pq|rs)
    by explicitly coding the FOUR sequential O(N^5) quarter-transformations.

    Parameters
    ----------
    eri_ao : np.ndarray, shape (n_ao, n_ao, n_ao, n_ao)
        Two-electron repulsion integrals (uv|ls) in chemist's notation.
    C : np.ndarray, shape (n_ao, n_mo)
        Canonical RHF MO coefficient matrix.

    Returns
    -------
    eri_mo : np.ndarray, shape (n_mo, n_mo, n_mo, n_mo)
        Two-electron integrals (pq|rs) in the spatial MO basis (chemist's notation).
    '''
    # TODO 2a: Quarter-transform 1: contract index 0 (u -> p) to form I1 of shape (n_mo, n_ao, n_ao, n_ao)
    # TODO 2b: Quarter-transform 2: contract index 1 (v -> q) to form I2 of shape (n_mo, n_mo, n_ao, n_ao)
    # TODO 2c: Quarter-transform 3: contract index 2 (l -> r) to form I3 of shape (n_mo, n_mo, n_mo, n_ao)
    # TODO 2d: Quarter-transform 4: contract index 3 (s -> s_mo) to form eri_mo of shape (n_mo, n_mo, n_mo, n_mo)
    raise NotImplementedError("Complete TODO 2: transform_eri_ao_to_mo_quarter_steps")


def build_spin_orbital_integrals(
    h_mo: np.ndarray, eri_mo: np.ndarray, eps: np.ndarray
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    '''
    TODO 3: Convert spatial MO integrals into the interleaved spin-orbital basis
    (2p = alpha, 2p+1 = beta) and return (h_so, g_so, eps_so).

    Parameters
    ----------
    h_mo : np.ndarray, shape (n_mo, n_mo)
        Spatial MO core Hamiltonian.
    eri_mo : np.ndarray, shape (n_mo, n_mo, n_mo, n_mo)
        Spatial MO 2e integrals (pq|rs) in chemist's notation.
    eps : np.ndarray, shape (n_mo,)
        Spatial RHF orbital energies.

    Returns
    -------
    h_so : np.ndarray, shape (2*n_mo, 2*n_mo)
        Spin-orbital 1e core Hamiltonian (block-diagonal in spin).
    g_so : np.ndarray, shape (2*n_mo, 2*n_mo, 2*n_mo, 2*n_mo)
        Antisymmetrized physicist's spin-orbital integrals <pq||rs> = <pq|rs> - <pq|sr>.
    eps_so : np.ndarray, shape (2*n_mo,)
        Spin-orbital energies [eps_0, eps_0, eps_1, eps_1, ...].
    '''
    # TODO 3a: Allocate h_so, eri_phys, and eps_so for n_so = 2 * n_mo.
    # TODO 3b: Populate eps_so and h_so (remembering <p sigma | h | q sigma'> = h_{pq} delta_{sigma, sigma'}).
    # TODO 3c: Populate physicist's <pq|rs> = (pr|qs) * delta(spin_p, spin_r) * delta(spin_q, spin_s).
    #          Watch the index transpose from chemist's (pr|qs) to physicist's <pq|rs>!
    # TODO 3d: Antisymmetrize: g_so = eri_phys - eri_phys.transpose(0, 1, 3, 2).
    raise NotImplementedError("Complete TODO 3: build_spin_orbital_integrals")


def compute_spin_orbital_rhf_energy(
    h_so: np.ndarray, g_so: np.ndarray, e_nuc: float, n_occ_so: int
) -> float:
    '''
    TODO 4: Compute the total RHF energy from spin-orbital integrals over occupied spin orbitals 0..n_occ_so-1.
    '''
    # TODO 4: Implement E_RHF = sum_{i < n_occ_so} h_so[i, i] + 0.5 * sum_{i,j < n_occ_so} g_so[i, j, i, j] + e_nuc
    raise NotImplementedError("Complete TODO 4: compute_spin_orbital_rhf_energy")

## Part 2: Second-Order Møller–Plesset Perturbation Theory (MP2, SCS-MP2 & 1-RDM)

### Mathematical Specifications for TODOs 5–8
1. **Four-Index Energy Denominators (`build_mp2_denominators`)**:
   For doubly occupied spatial orbitals $i, j \in \{0, \dots, n_{\text{occ}}-1\}$ and virtual spatial orbitals $a, b \in \{n_{\text{occ}}, \dots, N_{\text{MO}}-1\}$ (with $n_{\text{virt}} = N_{\text{MO}} - n_{\text{occ}}$), construct the 4D denominator tensor of shape `(n_occ, n_virt, n_occ, n_virt)` matching the $(ia|jb)$ index layout:
   $$\Delta_{iajb} = \varepsilon_i - \varepsilon_a + \varepsilon_j - \varepsilon_b$$
   *Constraint:* Use NumPy array broadcasting (`[:, None, None, None]`, etc.) — **no Python `for` loops**.

2. **Spatial-Orbital MP2, Opposite/Same-Spin Decomposition, SCS-MP2 & Pair Energies (`compute_spatial_mp2_and_scs`)**:
   Slice the `(n_occ, n_virt, n_occ, n_virt)` integral block $V_{iajb} = (ia|jb)$ from `eri_mo` and compute the first-order doubles amplitudes ([Psi4NumPy MP2 Tutorial](https://github.com/psi4/psi4numpy/blob/master/Tutorials/05_Moller-Plesset/5a_conventional-mp2.ipynb)):
   $$t_{iajb} = \frac{(ia|jb)}{\varepsilon_i - \varepsilon_a + \varepsilon_j - \varepsilon_b}$$
   Note that exchanging the two virtual indices $(a \leftrightarrow b)$ gives $(ib|ja)$, which corresponds to `np.swapaxes(iajb, 1, 3)`. Decompose the correlation energy into:
   - **Opposite-Spin ($\alpha\beta$) correlation**:
     $$E_{\text{OS}} = \sum_{i,j}^{\text{occ}}\sum_{a,b}^{\text{virt}} t_{iajb}\,(ia|jb)$$
   - **Same-Spin ($\alpha\alpha + \beta\beta$) correlation**:
     $$E_{\text{SS}} = \sum_{i,j}^{\text{occ}}\sum_{a,b}^{\text{virt}} t_{iajb}\,\big[(ia|jb) - (ib|ja)\big]$$
   - **Total Canonical MP2 & Grimme's SCS-MP2 ($c_{\text{OS}} = 1.20, c_{\text{SS}} = 1/3$)**:
     $$E_{\text{corr}}^{\text{MP2}} = E_{\text{OS}} + E_{\text{SS}}, \qquad E_{\text{corr}}^{\text{SCS-MP2}} = 1.20\,E_{\text{OS}} + \frac{1}{3}\,E_{\text{SS}}$$
   - **Occupied Pair-Correlation Matrix ($e_{ij}$ of shape `(n_occ, n_occ)`)**:
     $$e_{ij} = \sum_{a,b}^{\text{virt}} t_{iajb}\,\big[2(ia|jb) - (ib|ja)\big] \quad \left(\text{verify that } \sum_{i,j} e_{ij} = E_{\text{corr}}^{\text{MP2}}\right)$$

3. **Spin-Orbital MP2 Cross-Check (`compute_spin_orbital_mp2`)**:
   In the spin-orbital basis (with $N_{\text{occ}}^{\text{SO}} = 2 n_{\text{occ}}$), evaluate:
   $$E_{\text{corr}}^{\text{MP2, SO}} = \frac{1}{4}\sum_{i,j=0}^{N_{\text{occ}}^{\text{SO}}-1}\sum_{a,b=N_{\text{occ}}^{\text{SO}}}^{N_{\text{SO}}-1} \frac{|\langle ij \| ab \rangle|^2}{\varepsilon_i^{\text{SO}} + \varepsilon_j^{\text{SO}} - \varepsilon_a^{\text{SO}} - \varepsilon_b^{\text{SO}}}$$
   Why is the prefactor $\frac{1}{4}$ present? Because summing independently over all $i,j$ and $a,b$ double-counts both hole pairs $(i < j)$ and particle pairs $(a < b)$!

4. **Unrelaxed MP2 1-RDM & Natural Orbital Occupation Numbers (`compute_mp2_1rdm_and_noons`)**:
   In the canonical MO basis, the Hartree–Fock spin-summed 1-RDM is $\mathbf{P}^{\text{HF}} = \text{diag}(\underbrace{2, \dots, 2}_{n_{\text{occ}}}, \underbrace{0, \dots, 0}_{n_{\text{virt}}})$. At second order, electron correlation depletes the occupied orbitals and populates the virtual orbitals:
   $$\Delta P_{ik}^{\text{occ}} = -2 \sum_{j}^{\text{occ}}\sum_{a,b}^{\text{virt}} t_{iajb}\,\tilde{\tau}_{kajb}, \qquad \Delta P_{ab}^{\text{virt}} = +2 \sum_{i,j}^{\text{occ}}\sum_{c}^{\text{virt}} t_{iajc}\,\tilde{\tau}_{ibjc}$$
   where $\tilde{\tau}_{iajb} = 2 t_{iajb} - t_{ibja}$. Diagonalizing $\mathbf{P}^{\text{MP2}} = \mathbf{P}^{\text{HF}} + \Delta\mathbf{P}$ gives the **Natural Orbital Occupation Numbers (NOONs)** $n_\alpha \in [0, 2]$.

In [ ]:
def build_mp2_denominators(eps: np.ndarray, n_occ: int) -> np.ndarray:
    '''
    TODO 5: Build the 4D energy denominator tensor D_{iajb} = eps_i - eps_a + eps_j - eps_b
    of shape (n_occ, n_virt, n_occ, n_virt) using NumPy broadcasting (NO for-loops).
    '''
    # TODO 5: Slice e_occ = eps[:n_occ] and e_virt = eps[n_occ:], then broadcast into (n_occ, n_virt, n_occ, n_virt)
    raise NotImplementedError("Complete TODO 5: build_mp2_denominators")


def compute_spatial_mp2_and_scs(
    eri_mo: np.ndarray, eps: np.ndarray, n_occ: int
) -> dict[str, float | np.ndarray]:
    '''
    TODO 6: Compute spatial-orbital MP2 amplitudes, opposite-spin (e_os), same-spin (e_ss),
    total MP2 (e_mp2_corr), Grimme SCS-MP2 (e_scs_mp2_corr), and pair energy matrix (pair_energies).

    Returns
    -------
    dict with keys:
        "t2": np.ndarray of shape (n_occ, n_virt, n_occ, n_virt)
        "e_os": float
        "e_ss": float
        "e_mp2_corr": float
        "e_scs_mp2_corr": float
        "pair_energies": np.ndarray of shape (n_occ, n_occ)
    '''
    # TODO 6a: Slice iajb = eri_mo[:n_occ, n_occ:, :n_occ, n_occ:] and form ibja via swapaxes.
    # TODO 6b: Call build_mp2_denominators(eps, n_occ) and form t2 = iajb / denom.
    # TODO 6c: Evaluate e_os, e_ss, e_mp2_corr, e_scs_mp2_corr, and pair_energies (via np.einsum).
    raise NotImplementedError("Complete TODO 6: compute_spatial_mp2_and_scs")


def compute_spin_orbital_mp2(g_so: np.ndarray, eps_so: np.ndarray, n_occ_so: int) -> float:
    '''
    TODO 7: Compute the MP2 correlation energy in the spin-orbital basis:
    E_MP2 = (1/4) * sum_{ijab} |<ij||ab>|^2 / (eps_i + eps_j - eps_a - eps_b).
    '''
    # TODO 7: Slice g_oovv = g_so[:n_occ_so, :n_occ_so, n_occ_so:, n_occ_so:], construct the
    #         (n_occ_so, n_occ_so, n_virt_so, n_virt_so) denominator, and return 0.25 * sum(g_oovv^2 / denom).
    raise NotImplementedError("Complete TODO 7: compute_spin_orbital_mp2")


def compute_mp2_1rdm_and_noons(t2: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    '''
    TODO 8: Construct the unrelaxed MP2 spin-summed 1-RDM P_mp2 in the MO basis and
    return (P_mp2, sorted_noons_descending).

    Parameters
    ----------
    t2 : np.ndarray, shape (n_occ, n_virt, n_occ, n_virt)
        Spatial MP2 amplitudes t_{iajb}.

    Returns
    -------
    P_mp2 : np.ndarray, shape (n_mo, n_mo)
        Correlated 1-RDM in the spatial MO basis (trace must equal 2 * n_occ).
    noons : np.ndarray, shape (n_mo,)
        Natural Orbital Occupation Numbers sorted in descending order.
    '''
    # TODO 8a: Determine n_occ, n_virt from t2.shape and form tau_tilde = 2.0 * t2 - np.swapaxes(t2, 1, 3).
    # TODO 8b: Initialize P_mp2 = diag(2, ..., 2, 0, ..., 0).
    # TODO 8c: Subtract occupied depletion 2 * einsum("iajb,kajb->ik", t2, tau_tilde) from P_mp2[:n_occ, :n_occ]
    #          and add virtual population 2 * einsum("iajc,ibjc->ab", t2, tau_tilde) to P_mp2[n_occ:, n_occ:].
    # TODO 8d: Diagonalize P_mp2 with eigh and return (P_mp2, eigenvalues sorted descending).
    raise NotImplementedError("Complete TODO 8: compute_mp2_1rdm_and_noons")

## Part 3: Configuration Interaction — Determinants, Slater–Condon Rules & CISD

### Mathematical Specifications for TODOs 9–13
To understand how Configuration Interaction works at the determinant level, you will construct the **Slater determinant basis** and evaluate Hamiltonian matrix elements $\langle D_I | \hat{H}_{\text{elec}} | D_J \rangle$ using the **Slater–Condon rules**:

1. **Generating the $S_z = 0$ CISD Determinant Basis (`generate_sz0_cisd_determinants`)**:
   Represent each $N_{\text{elec}}$-electron Slater determinant as a **sorted tuple of occupied spin-orbital indices**:
   - Reference determinant: $|\Phi_0\rangle = (0, 1, \dots, N_{\text{occ}}^{\text{SO}} - 1)$.
   - Single excitations $|\Phi_i^a\rangle$: For each $i \in \text{occ}$ and $a \in \text{virt}$ with the **same spin** (`(i % 2) == (a % 2)`), replace $i$ by $a$ and sort the resulting tuple.
   - Double excitations $|\Phi_{ij}^{ab}\rangle$: For each pair $i < j$ in $\text{occ}$ and $a < b$ in $\text{virt}$ conserving total $S_z$ (`(i % 2) + (j % 2) == (a % 2) + (b % 2)`), replace $(i, j)$ by $(a, b)$ and sort the resulting tuple.

2. **Excitation Differences & Fermionic Permutation Parity (`align_determinants_and_parity`)**:
   Given two sorted tuples `det_bra` and `det_ket` of length $N_{\text{elec}}$:
   - Find `diff_bra = [p for p in det_bra if p not in det_ket]` (orbitals annihilated in `det_bra`)
   - Find `diff_ket = [r for r in det_ket if r not in det_bra]` (orbitals created in `det_ket`)
   - Find `common = [q for q in det_bra if q in det_ket]`
   - If `len(diff_bra)` is $1$ or $2$, moving the differing orbitals in `det_bra` to the front while keeping `common` in sorted order requires $\sum_{p \in \text{diff\_bra}} \text{det\_bra.index}(p)$ swaps (up to an even offset for $n_{\text{diff}}=2$), and similarly for `det_ket`. Therefore, the exact fermionic alignment sign is:
     $$\text{sign} = (-1)^{\sum_{p \in \text{diff\_bra}} \text{index}_{\text{bra}}(p) \;+\; \sum_{r \in \text{diff\_ket}} \text{index}_{\text{ket}}(r)}$$

3. **Slater–Condon Matrix Element (`slater_condon_element`)**:
   Using `align_determinants_and_parity(det_bra, det_ket)`:
   - **$n_{\text{diff}} == 0$**: $\sum_{p \in D_I} h_{pp}^{\text{SO}} + \frac{1}{2}\sum_{p \in D_I}\sum_{q \in D_I} \langle pq \| pq \rangle$ *(Careful: sum over pairs $(p,q)$ of `g_so[p, q, p, q]`, not all 4 indices independently!)*
   - **$n_{\text{diff}} == 1$ ($p \to r$)**: $\text{sign} \times \left( h_{pr}^{\text{SO}} + \sum_{q \in \text{common}} \langle pq \| rq \rangle \right)$
   - **$n_{\text{diff}} == 2$ ($p, q \to r, s$)**: $\text{sign} \times \langle pq \| rs \rangle$
   - **$n_{\text{diff}} \ge 3$**: $0.0$

4. **CISD Matrix Assembly & Diagonalization (`run_cisd_solver`)**:
   Build the symmetric matrix $H_{IJ}^{\text{CISD}} = \langle D_I | \hat{H}_{\text{elec}} | D_J \rangle$, verify **Brillouin's theorem** ($\|\mathbf{H}_{0, \text{singles}}^{\text{CISD}}\|_2 \approx 0$), diagonalize with `scipy.linalg.eigh`, and return the ground-state energy $E_{\text{CISD}} = \lambda_0 + E_{\text{nuc}}$, correlation energy $E_{\text{corr}}^{\text{CISD}} = E_{\text{CISD}} - (H_{00} + E_{\text{nuc}})$, and reference weight $|c_0|^2$.

5. **Exact 2-Electron Singlet Full CI (`solve_2e_singlet_fci`)**:
   For a 2-electron singlet system ($\text{HeH}^+$), the exact spatial wavefunction in the product basis $|\phi_p(\mathbf{r}_1)\alpha\,\phi_q(\mathbf{r}_2)\beta\rangle$ is governed by the $(N_{\text{MO}}^2 \times N_{\text{MO}}^2)$ Hamiltonian:
   $$\mathbf{H}_{2e} = \mathbf{h}^{\text{MO}} \otimes \mathbf{I} + \mathbf{I} \otimes \mathbf{h}^{\text{MO}} + \text{reshape}\big(\text{transpose}((pr|qs), (0, 2, 1, 3)),\ (N_{\text{MO}}^2, N_{\text{MO}}^2)\big)$$
   Diagonalize $\mathbf{H}_{2e}$ to obtain $E_{\text{FCI}}$, reshape the ground-state eigenvector into the $(N_{\text{MO}}, N_{\text{MO}})$ coefficient matrix $C_{pq}$, and form the exact spin-summed 1-RDM $P_{pq} = 2(\mathbf{C}\mathbf{C}^\mathsf{T})_{pq}$ and 2-RDM $\Gamma_{pqrs} = 2 C_{pr} C_{qs}$.

In [ ]:
def generate_sz0_cisd_determinants(
    n_occ_so: int, n_so: int
) -> tuple[list[tuple[int, ...]], int, int]:
    '''
    TODO 9: Generate the Sz=0 CISD determinant list [ref_det, ...singles..., ...doubles...]
    and return (determinants, n_singles, n_doubles).
    '''
    # TODO 9a: Form ref_det = tuple(range(n_occ_so)).
    # TODO 9b: Generate all spin-conserving single excitations (i % 2 == a % 2), sorted as tuples.
    # TODO 9c: Generate all spin-conserving double excitations (i < j, a < b, (i%2)+(j%2) == (a%2)+(b%2)), sorted as tuples.
    #          Return (dets, n_singles, n_doubles).
    raise NotImplementedError("Complete TODO 9: generate_sz0_cisd_determinants")


def align_determinants_and_parity(
    det_bra: tuple[int, ...], det_ket: tuple[int, ...]
) -> tuple[int, list[int], list[int], list[int], int]:
    '''
    TODO 10: Compare two sorted determinant tuples det_bra and det_ket.

    Returns
    -------
    n_diff : int
        Number of spin orbitals by which det_bra and det_ket differ.
    diff_bra : list[int]
        Spin orbitals present in det_bra but absent in det_ket.
    diff_ket : list[int]
        Spin orbitals present in det_ket but absent in det_bra.
    common : list[int]
        Spin orbitals present in both det_bra and det_ket.
    sign : int
        Fermionic permutation parity (+1 or -1) required to align differing orbitals.
    '''
    # TODO 10: Compute diff_bra, diff_ket, common, n_diff, and permutation parity sign = (-1)**(sum(pos_bra) + sum(pos_ket)).
    raise NotImplementedError("Complete TODO 10: align_determinants_and_parity")


def slater_condon_element(
    det_bra: tuple[int, ...],
    det_ket: tuple[int, ...],
    h_so: np.ndarray,
    g_so: np.ndarray,
) -> float:
    '''
    TODO 11: Evaluate the electronic Hamiltonian matrix element <det_bra | H_elec | det_ket>
    using the Slater-Condon rules and `align_determinants_and_parity`.
    '''
    # TODO 11: Call align_determinants_and_parity(det_bra, det_ket) and handle:
    #          - n_diff >= 3 -> 0.0
    #          - n_diff == 0 -> sum_{p} h_so[p,p] + 0.5 * sum_{p,q} g_so[p,q,p,q]
    #          - n_diff == 1 -> sign * (h_so[p,r] + sum_{q in common} g_so[p,q,r,q])
    #          - n_diff == 2 -> sign * g_so[p,q,r,s]
    raise NotImplementedError("Complete TODO 11: slater_condon_element")


def run_cisd_solver(
    h_so: np.ndarray, g_so: np.ndarray, e_nuc: float, n_occ_so: int
) -> dict[str, float | int | np.ndarray]:
    '''
    TODO 12: Build the CISD Hamiltonian matrix using `generate_sz0_cisd_determinants` and
    `slater_condon_element`, diagonalize it, and return a dictionary with keys:
    'n_determinants', 'n_singles', 'n_doubles', 'e_cisd_tot', 'e_cisd_corr',
    'c0', 'c0_weight', 'brillouin_norm', 'H_cisd', 'ci_vector'.
    '''
    # TODO 12: Assemble symmetric H_cisd, compute brillouin_norm = ||H_cisd[0, 1:1+n_singles]||_2,
    #          diagonalize H_cisd with scipy.linalg.eigh, and extract the ground-state properties.
    raise NotImplementedError("Complete TODO 12: run_cisd_solver")


def solve_2e_singlet_fci(
    h_mo: np.ndarray, eri_mo: np.ndarray, e_nuc: float
) -> tuple[float, np.ndarray, np.ndarray]:
    '''
    TODO 13: Solve the exact 2-electron singlet Full CI eigenvalue problem in the spatial MO basis
    and return (e_fci_total, rdm1_mo, rdm2_mo).
    '''
    # TODO 13: Build H_2e = kron(h_mo, I) + kron(I, h_mo) + eri_mo.transpose(0, 2, 1, 3).reshape(n_mo^2, n_mo^2),
    #          diagonalize with eigh, symmetrize and normalize C_mat, and return (e_elec + e_nuc, rdm1, rdm2).
    raise NotImplementedError("Complete TODO 13: solve_2e_singlet_fci")

## Part 4: Coupled Cluster Singles and Doubles (Spin-Orbital CCSD Step-by-Step)

### Mathematical Specifications for TODOs 14–19
In spin-orbital CCSD ([Stanton, Gauss, Watts & Bartlett, *J. Chem. Phys.* **94**, 4334 (1991)](https://pycrawfordprogproj.readthedocs.io/en/latest/Project_05/Project_05.html); [Psi4NumPy Spin-Orbital CCSD](https://github.com/psi4/psi4numpy/blob/master/Coupled-Cluster/Spin_Orbitals/CCSD/CCSD_DIIS.py)), we partition spin orbitals into occupied slice `o = slice(0, n_occ_so)` (indices $i, j, m, n$) and virtual slice `v = slice(n_occ_so, n_so)` (indices $a, b, e, f$). In canonical RHF orbitals, the spin-orbital Fock matrix is $f_{pq} = \varepsilon_p^{\text{SO}}\delta_{pq}$.

You will implement **six modular functions** corresponding to the exact algebraic blocks of CCSD:

1. **Effective Doubles Tensors (`build_ccsd_tau_tensors` — TODO 14)**:
   $$\tilde{\tau}_{ij}^{ab} = t_{ij}^{ab} + \frac{1}{2}\left(t_i^a t_j^b - t_i^b t_j^a\right), \qquad \tau_{ij}^{ab} = t_{ij}^{ab} + \left(t_i^a t_j^b - t_i^b t_j^a\right)$$

2. **One-Body Effective Fock Intermediates (`build_ccsd_F_intermediates` — TODO 15)**:
   $$\begin{aligned}
   \mathcal{F}_{ae} &= (1 - \delta_{ae})f_{ae} - \frac{1}{2}\sum_m f_{me} t_m^a + \sum_{mf} t_m^f \langle ma \| fe \rangle - \frac{1}{2}\sum_{mnf} \tilde{\tau}_{mn}^{af} \langle mn \| ef \rangle \\
   \mathcal{F}_{mi} &= (1 - \delta_{mi})f_{mi} + \frac{1}{2}\sum_e t_i^e f_{me} + \sum_{ne} t_n^e \langle mn \| ie \rangle + \frac{1}{2}\sum_{nef} \tilde{\tau}_{in}^{ef} \langle mn \| ef \rangle \\
   \mathcal{F}_{me} &= f_{me} + \sum_{nf} t_n^f \langle mn \| ef \rangle
   \end{aligned}$$

3. **Two-Body Effective Interaction Intermediates (`build_ccsd_W_intermediates` — TODO 16)**:
   $$\begin{aligned}
   \mathcal{W}_{mnij} &= \langle mn \| ij \rangle + \hat{P}(ij)\sum_e t_j^e \langle mn \| ie \rangle + \frac{1}{4}\sum_{ef} \tau_{ij}^{ef} \langle mn \| ef \rangle \\
   \mathcal{W}_{abef} &= \langle ab \| ef \rangle - \hat{P}(ab)\sum_m t_m^b \langle am \| ef \rangle + \frac{1}{4}\sum_{mn} \tau_{mn}^{ab} \langle mn \| ef \rangle \\
   \mathcal{W}_{mbej} &= \langle mb \| ej \rangle + \sum_f t_j^f \langle mb \| ef \rangle - \sum_n t_n^b \langle mn \| ej \rangle - \sum_{nf}\left(\frac{1}{2}t_{jn}^{fb} + t_j^f t_n^b\right)\langle mn \| ef \rangle
   \end{aligned}$$
   where the antisymmetrizer $\hat{P}(ij)X_{ij} = X_{ij} - X_{ji}$ swaps indices $i \leftrightarrow j$, and $\hat{P}(ab)X_{ab} = X_{ab} - X_{ba}$ swaps $a \leftrightarrow b$.

4. **Singles ($T_1$) Amplitude Update (`update_t1_amplitudes` — TODO 17)**:
   With $D_i^a = \varepsilon_i^{\text{SO}} - \varepsilon_a^{\text{SO}}$:
   $$D_i^a\,t_i^{a\,\text{(new)}} = f_{ia} + \sum_e t_i^e \mathcal{F}_{ae} - \sum_m t_m^a \mathcal{F}_{mi} + \sum_{me} t_{im}^{ae} \mathcal{F}_{me} - \sum_{nf} t_n^f \langle na \| if \rangle - \frac{1}{2}\sum_{mef} t_{im}^{ef} \langle ma \| ef \rangle - \frac{1}{2}\sum_{mne} t_{mn}^{ae} \langle nm \| ei \rangle$$

5. **Doubles ($T_2$) Amplitude Update (`update_t2_amplitudes` — TODO 18)**:
   With $D_{ij}^{ab} = \varepsilon_i^{\text{SO}} + \varepsilon_j^{\text{SO}} - \varepsilon_a^{\text{SO}} - \varepsilon_b^{\text{SO}}$:
   $$\begin{aligned}
   D_{ij}^{ab}\,t_{ij}^{ab\,\text{(new)}} &= \langle ij \| ab \rangle + \hat{P}(ab)\sum_e t_{ij}^{ae}\left(\mathcal{F}_{be} - \frac{1}{2}\sum_m t_m^b \mathcal{F}_{me}\right) - \hat{P}(ij)\sum_m t_{im}^{ab}\left(\mathcal{F}_{mj} + \frac{1}{2}\sum_e t_j^e \mathcal{F}_{me}\right) \\
   &\quad + \frac{1}{2}\sum_{mn} \tau_{mn}^{ab}\mathcal{W}_{mnij} + \frac{1}{2}\sum_{ef} \tau_{ij}^{ef}\mathcal{W}_{abef} + \hat{P}(ij)\hat{P}(ab)\sum_{me}\left(t_{im}^{ae}\mathcal{W}_{mbej} - t_i^e t_m^a \langle mb \| ej \rangle\right) \\
   &\quad + \hat{P}(ij)\sum_e t_i^e \langle ab \| ej \rangle - \hat{P}(ab)\sum_m t_m^a \langle mb \| ij \rangle
   \end{aligned}$$
   *(Note: Applying $\hat{P}(ij)\hat{P}(ab)$ to a tensor $Y_{ijab}$ means $Y_{ijab} - Y_{jiab} - Y_{ijba} + Y_{jiba}$.)*

6. **CCSD Energy & Iterative Driver (`run_ccsd_driver` — TODO 19)**:
   At each step, evaluate:
   $$E_{\text{corr}}^{\text{CCSD}} = \sum_{ia} f_{ia}\,t_i^a + \frac{1}{4}\sum_{ijab} \langle ij \| ab \rangle\,\tau_{ij}^{ab}$$
   and upon convergence compute the **Lee–Taylor $T_1$ diagnostic** $\mathcal{T}_1 = \frac{\|\mathbf{t}_1\|_F}{\sqrt{2 N_{\text{elec}}}}$.

In [ ]:
def build_ccsd_tau_tensors(t1: np.ndarray, t2: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    '''
    TODO 14: Construct the effective doubles intermediates (tau_tilde, tau) from t1 and t2.

    Parameters
    ----------
    t1 : np.ndarray, shape (n_occ_so, n_virt_so)
    t2 : np.ndarray, shape (n_occ_so, n_occ_so, n_virt_so, n_virt_so)

    Returns
    -------
    tau_tilde, tau : tuple of np.ndarray, each of shape (n_occ_so, n_occ_so, n_virt_so, n_virt_so)
    '''
    # TODO 14: Form antisymmetrized outer product t1_t1 = einsum("ia,jb->ijab", t1, t1) - einsum("ib,ja->ijab", t1, t1)
    #          and return (t2 + 0.5 * t1_t1, t2 + t1_t1).
    raise NotImplementedError("Complete TODO 14: build_ccsd_tau_tensors")


def build_ccsd_F_intermediates(
    f: np.ndarray,
    g: np.ndarray,
    t1: np.ndarray,
    tau_tilde: np.ndarray,
    o: slice,
    v: slice,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    '''
    TODO 15: Build the three Stanton-Gauss 1-body intermediates (F_ae, F_mi, F_me).
    '''
    # TODO 15: Implement F_ae (n_virt, n_virt), F_mi (n_occ, n_occ), and F_me (n_occ, n_virt)
    #          using the exact Stanton-Gauss formulas in the Markdown specification above.
    raise NotImplementedError("Complete TODO 15: build_ccsd_F_intermediates")


def build_ccsd_W_intermediates(
    g: np.ndarray,
    t1: np.ndarray,
    t2: np.ndarray,
    tau: np.ndarray,
    o: slice,
    v: slice,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    '''
    TODO 16: Build the three Stanton-Gauss 2-body intermediates (W_mnij, W_abef, W_mbej).
    '''
    # TODO 16: Implement W_mnij (o,o,o,o), W_abef (v,v,v,v), and W_mbej (o,v,v,o)
    #          using np.einsum and remembering the permutation operators P(ij) and P(ab).
    raise NotImplementedError("Complete TODO 16: build_ccsd_W_intermediates")


def update_t1_amplitudes(
    f: np.ndarray,
    g: np.ndarray,
    t1: np.ndarray,
    t2: np.ndarray,
    F_ae: np.ndarray,
    F_mi: np.ndarray,
    F_me: np.ndarray,
    D_ia: np.ndarray,
    o: slice,
    v: slice,
) -> np.ndarray:
    '''
    TODO 17: Compute the updated Singles amplitudes t1_new = rhs_t1 / D_ia.
    '''
    # TODO 17: Sum the 7 terms of rhs_t1 and divide elementwise by D_ia.
    raise NotImplementedError("Complete TODO 17: update_t1_amplitudes")


def update_t2_amplitudes(
    g: np.ndarray,
    t1: np.ndarray,
    t2: np.ndarray,
    tau: np.ndarray,
    F_ae: np.ndarray,
    F_mi: np.ndarray,
    F_me: np.ndarray,
    W_mnij: np.ndarray,
    W_abef: np.ndarray,
    W_mbej: np.ndarray,
    D_ijab: np.ndarray,
    o: slice,
    v: slice,
) -> np.ndarray:
    '''
    TODO 18: Compute the updated Doubles amplitudes t2_new = rhs_t2 / D_ijab.
    '''
    # TODO 18: Evaluate each term of rhs_t2, apply the antisymmetrizers P(ab), P(ij), and P(ij)P(ab),
    #          and divide elementwise by D_ijab.
    raise NotImplementedError("Complete TODO 18: update_t2_amplitudes")


def run_ccsd_driver(
    h_so: np.ndarray,
    g_so: np.ndarray,
    eps_so: np.ndarray,
    n_occ_so: int,
    max_iter: int = 80,
    tol: float = 1e-10,
) -> dict[str, float | np.ndarray | dict]:
    '''
    TODO 19: Run the complete spin-orbital CCSD iteration loop using TODOs 14-18.

    Returns
    -------
    dict with keys:
        'converged': bool
        'e_ccsd_corr': float
        't1': np.ndarray
        't2': np.ndarray
        't1_diagnostic': float
        'n_iterations': int
        'history': dict with lists 'iter', 'e_corr', 'delta_e', 'rms_t2'
    '''
    # TODO 19a: Set o = slice(0, n_occ_so), v = slice(n_occ_so, len(eps_so)), f = np.diag(eps_so),
    #           and construct D_ia and D_ijab from eps_so.
    # TODO 19b: Initialize t1 = 0 and t2 = g_so[o, o, v, v] / D_ijab (the MP2 starting guess).
    # TODO 19c: Loop for 1..max_iter: call TODOs 14, 15, 16, 17, 18, evaluate E_corr(CCSD),
    #           record history, and check convergence |e_corr_new - e_corr_old| < tol.
    raise NotImplementedError("Complete TODO 19: run_ccsd_driver")

## Part 5: Size-Extensivity Analysis & Diagnostic Visualization

### Mathematical Specifications for TODO 20
Consider $M$ identical, non-interacting closed-shell monomers separated by $R \to \infty$. Each monomer has correlation energy $E_{\text{corr}}(1)$ and (for CISD) reference determinant coefficient $c_0 = \cos\theta$, so $\tan^2\theta = \frac{1 - c_0^2}{c_0^2}$.
- Because MP2 and CCSD contain only linked diagrams, their supermolecule correlation energy is strictly additive:
  $$E_{\text{corr}}^{\text{MP2}}(M) = M\,E_{\text{corr}}^{\text{MP2}}(1), \qquad E_{\text{corr}}^{\text{CCSD}}(M) = M\,E_{\text{corr}}^{\text{CCSD}}(1)$$
- For truncated CISD, omitting simultaneous pair excitations on distinct monomers ($\hat{C}_2^{(A)}\hat{C}_2^{(B)}$) causes the supermolecule secular equation to yield:
  $$E_{\text{corr}}^{\text{CISD}}(M) = E_{\text{corr}}^{\text{CISD}}(1) \times \frac{\sqrt{1 + 4 M \tan^2\theta} - 1}{\sqrt{1 + 4\tan^2\theta} - 1}$$
  so the fraction of correlation energy recovered per monomer, $\frac{E_{\text{corr}}^{\text{CISD}}(M)}{M\,E_{\text{corr}}^{\text{CISD}}(1)}$, decays as $M^{-1/2} \to 0$ as $M \to \infty$!

In [ ]:
def compute_size_extensivity_scaling(
    e_corr_mp2_1: float,
    e_corr_ccsd_1: float,
    e_corr_cisd_1: float,
    c0_1: float,
    max_monomers: int = 10,
) -> dict[str, np.ndarray | float]:
    '''
    TODO 20: Compute the size-extensivity scaling across M = 1..max_monomers non-interacting monomers.

    Returns
    -------
    dict with keys:
        'M': np.ndarray of shape (max_monomers,) with values 1..max_monomers
        'mp2_per_monomer_ratio': np.ndarray of ones
        'ccsd_per_monomer_ratio': np.ndarray of ones
        'cisd_per_monomer_ratio': np.ndarray of E_CISD(M) / (M * E_CISD(1))
        'cisd_dimer_error_eh': float, E_CISD(M=2) - 2 * E_CISD(1)
    '''
    # TODO 20: Compute tan2 = (1 - c0_1**2) / (c0_1**2), evaluate cisd_per_monomer_ratio for M = 1..max_monomers,
    #          and compute cisd_dimer_error_eh for M = 2.
    raise NotImplementedError("Complete TODO 20: compute_size_extensivity_scaling")

## Part 6: Step-by-Step Checkpoint & Verification Suite

Run the cell below as you implement each `# TODO` function. It tests **each of the 20 functions in sequence** against exact numerical benchmarks on $\text{HeH}^+$ (`STO-3G`, 2 electrons) and $\text{H}_2\text{O}$ (`STO-3G`, 10 electrons), reporting your progress and automatically generating the 3-panel diagnostic figure once all 20 checkpoints pass!

### Target Reference Values ($\text{H}_2\text{O}$ / `STO-3G` & $\text{HeH}^+$ / `STO-3G`)
| Quantity | $\text{HeH}^+$ (`2e`, `2 AOs`) Target | $\text{H}_2\text{O}$ (`10e`, `7 AOs`) Target |
| :--- | :--- | :--- |
| **$E_{\text{RHF}}$ (Total Energy)** | $-1.6853183839\ E_{\text{h}}$ | $-74.9630620833\ E_{\text{h}}$ |
| **$E_{\text{corr}}^{\text{MP2}}$ ($E_{\text{OS}} + E_{\text{SS}}$)** | $-0.0105084097\ E_{\text{h}}$ ($100\%$ $E_{\text{OS}}$) | $-0.0355667899\ E_{\text{h}}$ ($E_{\text{OS}} = -0.03353419$, $E_{\text{SS}} = -0.00203260$) |
| **$E_{\text{corr}}^{\text{SCS-MP2}}$** | $-0.0126100916\ E_{\text{h}}$ | $-0.0409185571\ E_{\text{h}}$ |
| **$E_{\text{corr}}^{\text{CISD}}$ ($S_z=0$ Determinants)** | $-0.0163476089\ E_{\text{h}}$ ($4$ dets, exact FCI) | $-0.0488780286\ E_{\text{h}}$ ($141$ dets, $\|c_0\|^2 = 0.974490$) |
| **$E_{\text{corr}}^{\text{CCSD}}$ (Spin-Orbital CCSD)** | $-0.0163476089\ E_{\text{h}}$ (exact FCI) | $-0.0494674382\ E_{\text{h}}$ ($\mathcal{T}_1 = 0.00433$, 25 iters) |
| **CISD Non-Interacting Dimer Error ($M=2$)** | $+0.00018095\ E_{\text{h}}$ | $+0.00226657\ E_{\text{h}}$ ($\approx +1.42\text{ kcal/mol}$) |

In [2]:
def run_all_checkpoints() -> None:
    '''Step-by-step checkpoint verifier for all 20 #TODO functions in Section 5.'''
    passed = 0
    total_todos = 20

    try:
        # --- PART 1 CHECKS (TODOs 1-4) ---
        h_mo_heh = transform_1e_ao_to_mo(heh_data["T"] + heh_data["V"], rhf_heh.mo_coeff)
        h_mo_h2o = transform_1e_ao_to_mo(h2o_data["T"] + h2o_data["V"], rhf_h2o.mo_coeff)
        assert h_mo_h2o.shape == (7, 7) and np.isclose(h_mo_h2o[0, 0], -32.68529699, atol=1e-6)
        passed += 1
        print("[PASS] TODO 1 : transform_1e_ao_to_mo")

        eri_mo_heh = transform_eri_ao_to_mo_quarter_steps(heh_data["eri"], rhf_heh.mo_coeff)
        eri_mo_h2o = transform_eri_ao_to_mo_quarter_steps(h2o_data["eri"], rhf_h2o.mo_coeff)
        assert eri_mo_h2o.shape == (7, 7, 7, 7) and np.isclose(eri_mo_h2o[0, 0, 0, 0], 4.74905525, atol=1e-6)
        passed += 1
        print("[PASS] TODO 2 : transform_eri_ao_to_mo_quarter_steps")

        h_so_heh, g_so_heh, eps_so_heh = build_spin_orbital_integrals(h_mo_heh, eri_mo_heh, rhf_heh.orbital_energies)
        h_so_h2o, g_so_h2o, eps_so_h2o = build_spin_orbital_integrals(h_mo_h2o, eri_mo_h2o, rhf_h2o.orbital_energies)
        assert g_so_h2o.shape == (14, 14, 14, 14)
        assert np.linalg.norm(g_so_h2o + g_so_h2o.transpose(1, 0, 2, 3)) < 1e-13, "<pq||rs> must be antisymmetric in (p,q)!"
        assert np.linalg.norm(g_so_h2o + g_so_h2o.transpose(0, 1, 3, 2)) < 1e-13, "<pq||rs> must be antisymmetric in (r,s)!"
        passed += 1
        print("[PASS] TODO 3 : build_spin_orbital_integrals")

        e_rhf_so_h2o = compute_spin_orbital_rhf_energy(h_so_h2o, g_so_h2o, float(h2o_data["e_nuc"]), n_occ_so=10)
        assert np.isclose(e_rhf_so_h2o, rhf_h2o.e_total, atol=1e-10)
        passed += 1
        print(f"[PASS] TODO 4 : compute_spin_orbital_rhf_energy (E_RHF = {e_rhf_so_h2o:.10f} Eh)")

        # --- PART 2 CHECKS (TODOs 5-8) ---
        denom_h2o = build_mp2_denominators(rhf_h2o.orbital_energies, n_occ=5)
        assert denom_h2o.shape == (5, 2, 5, 2) and np.all(denom_h2o < 0.0)
        passed += 1
        print("[PASS] TODO 5 : build_mp2_denominators")

        mp2_heh = compute_spatial_mp2_and_scs(eri_mo_heh, rhf_heh.orbital_energies, n_occ=1)
        mp2_h2o = compute_spatial_mp2_and_scs(eri_mo_h2o, rhf_h2o.orbital_energies, n_occ=5)
        assert np.isclose(mp2_h2o["e_os"], -0.0335341852, atol=1e-8)
        assert np.isclose(mp2_h2o["e_ss"], -0.0020326048, atol=1e-8)
        assert np.isclose(mp2_h2o["e_mp2_corr"], -0.0355667899, atol=1e-8)
        assert np.isclose(mp2_h2o["e_scs_mp2_corr"], -0.0409185571, atol=1e-8)
        assert np.isclose(np.sum(mp2_h2o["pair_energies"]), mp2_h2o["e_mp2_corr"], atol=1e-12)
        passed += 1
        print(f"[PASS] TODO 6 : compute_spatial_mp2_and_scs (E_MP2_corr = {mp2_h2o['e_mp2_corr']:+.10f} Eh)")

        e_mp2_so_h2o = compute_spin_orbital_mp2(g_so_h2o, eps_so_h2o, n_occ_so=10)
        assert np.isclose(e_mp2_so_h2o, mp2_h2o["e_mp2_corr"], atol=1e-12)
        passed += 1
        print("[PASS] TODO 7 : compute_spin_orbital_mp2 (matches spatial MP2 to < 1e-12 Eh)")

        P_mp2_h2o, noons_h2o = compute_mp2_1rdm_and_noons(mp2_h2o["t2"])
        assert np.isclose(np.trace(P_mp2_h2o), 10.0, atol=1e-12)
        assert np.all(np.diff(noons_h2o) <= 0.0) and np.isclose(noons_h2o[0], 1.999997, atol=1e-5)
        passed += 1
        print(f"[PASS] TODO 8 : compute_mp2_1rdm_and_noons (NOONs = {np.round(noons_h2o, 5)})")

        # --- PART 3 CHECKS (TODOs 9-13) ---
        dets_h2o, n_s_h2o, n_d_h2o = generate_sz0_cisd_determinants(n_occ_so=10, n_so=14)
        assert len(dets_h2o) == 141 and n_s_h2o == 20 and n_d_h2o == 120
        passed += 1
        print(f"[PASS] TODO 9 : generate_sz0_cisd_determinants ({len(dets_h2o)} Sz=0 determinants)")

        n_diff, d_bra, d_ket, comm, sgn = align_determinants_and_parity((0, 1), (1, 2))
        assert n_diff == 1 and d_bra == [0] and d_ket == [2] and comm == [1] and sgn == -1
        passed += 1
        print("[PASS] TODO 10: align_determinants_and_parity")

        sc_00 = slater_condon_element(dets_h2o[0], dets_h2o[0], h_so_h2o, g_so_h2o)
        assert np.isclose(sc_00 + float(h2o_data["e_nuc"]), rhf_h2o.e_total, atol=1e-10)
        passed += 1
        print("[PASS] TODO 11: slater_condon_element")

        cisd_heh = run_cisd_solver(h_so_heh, g_so_heh, float(heh_data["e_nuc"]), n_occ_so=2)
        cisd_h2o = run_cisd_solver(h_so_h2o, g_so_h2o, float(h2o_data["e_nuc"]), n_occ_so=10)
        assert cisd_h2o["brillouin_norm"] < 1e-9
        assert np.isclose(cisd_h2o["e_cisd_corr"], -0.0488780286, atol=1e-8)
        passed += 1
        print(f"[PASS] TODO 12: run_cisd_solver (E_CISD_corr = {cisd_h2o['e_cisd_corr']:+.10f} Eh)")

        fci_heh_tot, rdm1_heh, rdm2_heh = solve_2e_singlet_fci(h_mo_heh, eri_mo_heh, float(heh_data["e_nuc"]))
        assert np.isclose(fci_heh_tot, cisd_heh["e_cisd_tot"], atol=1e-9)
        assert np.isclose(np.trace(rdm1_heh), 2.0, atol=1e-12)
        passed += 1
        print(f"[PASS] TODO 13: solve_2e_singlet_fci (HeH+ E_FCI = {fci_heh_tot:.10f} Eh == E_CISD)")

        # --- PART 4 CHECKS (TODOs 14-19) ---
        o_s, v_s = slice(0, 10), slice(10, 14)
        f_s = np.diag(eps_so_h2o)
        D_ia_s = eps_so_h2o[o_s, None] - eps_so_h2o[None, v_s]
        D_ijab_s = (
            eps_so_h2o[o_s, None, None, None] + eps_so_h2o[None, o_s, None, None]
            - eps_so_h2o[None, None, v_s, None] - eps_so_h2o[None, None, None, v_s]
        )
        t1_0 = np.zeros((10, 4))
        t2_0 = g_so_h2o[o_s, o_s, v_s, v_s] / D_ijab_s

        tau_t, tau_f = build_ccsd_tau_tensors(t1_0, t2_0)
        assert tau_t.shape == (10, 10, 4, 4) and np.allclose(tau_t, t2_0)
        passed += 1
        print("[PASS] TODO 14: build_ccsd_tau_tensors")

        F_ae, F_mi, F_me = build_ccsd_F_intermediates(f_s, g_so_h2o, t1_0, tau_t, o_s, v_s)
        assert F_ae.shape == (4, 4) and F_mi.shape == (10, 10) and F_me.shape == (10, 4)
        passed += 1
        print("[PASS] TODO 15: build_ccsd_F_intermediates")

        W_mnij, W_abef, W_mbej = build_ccsd_W_intermediates(g_so_h2o, t1_0, t2_0, tau_f, o_s, v_s)
        assert W_mnij.shape == (10, 10, 10, 10) and W_abef.shape == (4, 4, 4, 4) and W_mbej.shape == (10, 4, 4, 10)
        passed += 1
        print("[PASS] TODO 16: build_ccsd_W_intermediates")

        t1_1 = update_t1_amplitudes(f_s, g_so_h2o, t1_0, t2_0, F_ae, F_mi, F_me, D_ia_s, o_s, v_s)
        assert t1_1.shape == (10, 4) and np.isclose(np.linalg.norm(t1_1), 0.00928228, atol=1e-5)
        passed += 1
        print("[PASS] TODO 17: update_t1_amplitudes")

        t2_1 = update_t2_amplitudes(g_so_h2o, t1_0, t2_0, tau_f, F_ae, F_mi, F_me, W_mnij, W_abef, W_mbej, D_ijab_s, o_s, v_s)
        assert t2_1.shape == (10, 10, 4, 4) and np.isclose(np.linalg.norm(t2_1), 0.2919947, atol=1e-5)
        passed += 1
        print("[PASS] TODO 18: update_t2_amplitudes")

        ccsd_heh = run_ccsd_driver(h_so_heh, g_so_heh, eps_so_heh, n_occ_so=2)
        ccsd_h2o = run_ccsd_driver(h_so_h2o, g_so_h2o, eps_so_h2o, n_occ_so=10)
        assert np.isclose(rhf_heh.e_total + ccsd_heh["e_ccsd_corr"], fci_heh_tot, atol=1e-9)
        assert np.isclose(ccsd_h2o["e_ccsd_corr"], -0.0494674382, atol=1e-8)
        passed += 1
        print(f"[PASS] TODO 19: run_ccsd_driver (E_CCSD_corr = {ccsd_h2o['e_ccsd_corr']:+.10f} Eh, T1 = {ccsd_h2o['t1_diagnostic']:.5f})")

        # --- PART 5 CHECKS (TODO 20) ---
        ext_h2o = compute_size_extensivity_scaling(
            mp2_h2o["e_mp2_corr"], ccsd_h2o["e_ccsd_corr"], cisd_h2o["e_cisd_corr"], cisd_h2o["c0"], max_monomers=10
        )
        assert np.isclose(ext_h2o["cisd_dimer_error_eh"], +0.00226657, atol=1e-6)
        passed += 1
        print(f"[PASS] TODO 20: compute_size_extensivity_scaling (CISD dimer error = {1000*ext_h2o['cisd_dimer_error_eh']:+.4f} mEh)")
        print(f"\nCONGRATULATIONS! All {passed}/{total_todos} Section 5 checkpoints passed!")

    except NotImplementedError as exc:
        print(f"\n[PROGRESS: {passed}/{total_todos} checkpoints passed] Next up -> {exc}")
    except AssertionError as exc:
        print(f"\n[ASSERTION FAILED after {passed}/{total_todos} checkpoints] -> {exc}")


# Uncomment and run as you implement each TODO:
# run_all_checkpoints()
print("Checkpoint suite loaded. Implement TODOs 1-20 above and call `run_all_checkpoints()` to test your code!")

Checkpoint suite loaded. Implement TODOs 1-20 above and call `run_all_checkpoints()` to test your code!
